# ClinicalTrials.gov analysis stratified by trial phase and status

Reviewer comment: a pair in ClinicalTrials.gov does not imply therapeutic validity; stratify by trial status or
outcome where feasible. ClinicalTrials.gov has no outcome label, but every trial has a **phase**: a drug-disease pair
that reached Phase 3 or 4 has passed earlier phases, which requires evidence of safety and preliminary efficacy
(Phase 4 = after approval). If NetMedGPT scores rise with the highest phase a pair reached, the scores reflect more
than entry into clinical investigation.

- Scores: the model and test pairs of `~/All_NetMedGPT/BioMedFormer/27-Clinical_trials_analysis.ipynb` (Fig. 4f-h),
  same code (cells 1-14), only the ray-tune lookup of the hyperparameters is replaced by the values of the checkpoint name.
- Labels as in that notebook: 0 = negative pairs, 1 = clinical-trial-only pairs, 2 = KG-supported trial pairs.
- Phase / status per pair: all ClinicalTrials.gov trials of the pair (`clinical_trial_data_with_drugbankmondo_ids.csv`,
  DrugBank and MONDO IDs mapped to the KG nodes); highest phase reached, and whether any trial was completed or stopped
  early (terminated / withdrawn / suspended).
- Per group: AUROC against the same negative pairs, median score, Mann-Whitney test; Spearman correlation of score and
  highest phase.

In [ ]:
import os, sys
BMF = "/home/bbc8731/All_NetMedGPT/BioMedFormer"
NETMEDGPT = "/home/bbc8731/NetMedGPT"
os.chdir(BMF)                       # the BioMedFormer code (src/) and data paths are relative to this folder
sys.path.insert(0, BMF)
OUT_DIR = os.path.join(NETMEDGPT, "data/new_analysis/clinical_trials_phase")
os.makedirs(OUT_DIR, exist_ok=True)

In [ ]:
import os
import subprocess
import json
from tqdm import tqdm
import time
import itertools
from datetime import datetime

import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
from torch.cuda.amp import autocast, GradScaler
import matplotlib.pyplot as plt
from scipy.stats import gaussian_kde
from sklearn.metrics import average_precision_score, precision_recall_curve
import seaborn as sns
from scipy.stats import mannwhitneyu
from collections import OrderedDict

from torch.optim.lr_scheduler import LambdaLR
import math

from src.utilities import *
# from src.model_pretrain import *
from src.model_pretrain import *

from src.similarity import *
from src.sentence_generation_transductive import *

In [ ]:
device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')

with open("/home/bbc8731/All_NetMedGPT/BioMedFormer/data/param/parameters.json", 'r') as file:
    all_param = json.load(file)

data_dir = os.path.join(all_param['files']['processed_data'], "pretrain_transductive")
best_param_dir = "//cosybio_project/faren/biomedformer/BioMedFormer_docker/data/ray_tune/log/train_model/"  
clinical_trials_results = "/home/bbc8731/All_NetMedGPT/BioMedFormer/data/plot/clinicaltrials"
clinical_trials_file_results = "/home/bbc8731/All_NetMedGPT/BioMedFormer/data/clinicaltrials/"

KG_dir = all_param['files']['KG_dir']
attr_dir = all_param['files']['node_attr']

test_data_dir = os.path.join(all_param['files']['processed_data'], "clinical_trials/random_splitted") 

os.makedirs(os.path.join(all_param['files']['data_dir'], 'result'), exist_ok = True) 
result_dir = os.path.join(all_param['files']['data_dir'], 'result')

os.makedirs(os.path.join(all_param['files']['data_dir'], "log"), exist_ok=True) # Ensure the folder exists
log_dir = os.path.join(all_param['files']['data_dir'], 'log')

os.makedirs(os.path.join(all_param['files']['data_dir'], "saved_models"), exist_ok=True) # Ensure the folder exists
model_dir = os.path.join(all_param['files']['data_dir'], 'saved_models')



In [ ]:
nodes = pd.read_csv(os.path.join(KG_dir, 'nodes.csv'), sep= ',')
edge = pd.read_csv(os.path.join(KG_dir, "edges_ctg_reduced.csv")) 
ctg = pd.read_csv(os.path.join(KG_dir, "edges_clinical_trials_with_positivesInPrimeKG.csv")) 
efficacy = pd.read_csv(os.path.join(KG_dir, "efficacy_preprocessed.csv")) 

all_ids = {
    'drug': torch.tensor(
        nodes.loc[nodes['node_type'] == 'drug', 'node_index'].unique(), dtype=torch.int64
    ),
    'gene': torch.tensor(
        nodes.loc[nodes['node_type'] == 'gene/protein', 'node_index'].unique(), dtype=torch.int64
    ),
    'phenotype': torch.tensor(
        nodes.loc[nodes['node_type'] == 'effect/phenotype', 'node_index'].unique(), dtype=torch.int64
    )
}

# Get mapping from node_types to z_index values
node_type_to_z = edge.groupby('node_types')['z_index'].unique().to_dict()

all_ids_node2 = {}

for z in node_type_to_z.get('disease|drug', []):
    all_ids_node2[z] = all_ids['drug']

for z in node_type_to_z.get('drug|gene/protein', []):
    all_ids_node2[z] = all_ids['gene']

for z in node_type_to_z.get('drug|effect/phenotype', []):
    all_ids_node2[z] = all_ids['phenotype']


# other hyperparameters
mask_token = edge['z_index'].max() +1
vocab_size = mask_token + 1

relation_type = list(edge['relation'].unique())
node_types = list(nodes['node_type'].unique())
node_types_without_feat = ['biological_process',
                           'molecular_function',
                           'cellular_component',
                           'exposure',
                           'pathway',
                           'anatomy']

mask = ['mask']
entity = node_types + relation_type + mask

without_feat = {
    'node': node_types_without_feat,
    'relation': relation_type,
    'mask': mask
}

## find relation_mask_index
relation_index = edge.loc[edge['relation'].isin(relation_type), ['relation', 'z_index']].drop_duplicates()
mask_row = pd.DataFrame([['mask', mask_token]], columns=['relation', 'z_index'])
relation_mask_index = pd.concat([relation_index, mask_row], ignore_index=True)
seq_len = (all_param['node2vec']['walk_length']*2)-1   # walk_length_with_relation


# hyperparameters of the checkpoint (file name: wl5_wpn30_dim300_head5_lencoder30_bs4000_lr0.0001), instead of the ray-tune lookup
best_hyperparam = {"hidden_channels": 300, "nhead": 5, "N_encoder_layers": 30, "batch_size": 4000, "learning_rate": 0.0001}

drug_pubmedbert = pd.read_csv(os.path.join(attr_dir, 'attr_drug_pubmedbert.csv'), sep= ',', index_col=0)
drug_pubmedbert = drug_pubmedbert.sort_index()
drug_pubmedbert = torch.tensor(drug_pubmedbert.values, dtype=torch.float32)  

drug_FP = pd.read_csv(os.path.join(attr_dir, 'attr_drug_fingerprint.csv'), sep= ',')
drug_FP = drug_FP.sort_index()
drug_FP = torch.tensor(drug_FP.values, dtype=torch.float32)  

gene_pubmedbert = pd.read_csv(os.path.join(attr_dir, 'attr_protein_pubmedbert.csv'), sep= ',', index_col=0)
gene_pubmedbert = gene_pubmedbert.sort_index()
gene_pubmedbert = torch.tensor(gene_pubmedbert.values, dtype=torch.float32)  

gene_esm2 = torch.load(os.path.join(attr_dir, 'attr_protein_emb_esm2.pt'))

disease_pubmedbert = pd.read_csv(os.path.join(attr_dir, 'attr_disease_pubmedbert.csv'), sep= ',', index_col = 0)
disease_pubmedbert = disease_pubmedbert.sort_index()
disease_pubmedbert = torch.tensor(disease_pubmedbert.values, dtype=torch.float32)  

phenotype_pubmedbert = pd.read_csv(os.path.join(attr_dir, 'attr_phenotype_pubmedbert.csv'), sep= ',', index_col = 0)
phenotype_pubmedbert = phenotype_pubmedbert.sort_index()
phenotype_pubmedbert = torch.tensor(phenotype_pubmedbert.values, dtype=torch.float32)  

# make a dict of all the features
feat = {
    'gene/protein': {
        'pubmedbert': gene_pubmedbert,
        'esm2': gene_esm2
    },
    'drug': {
        'pubmedbert': drug_pubmedbert,
        'FP': drug_FP
    },
    'disease': {
        'pubmedbert': disease_pubmedbert,
    },
    'effect/phenotype': {
        'pubmedbert': phenotype_pubmedbert,
    }
}

feat_rest = {}
for k, v in without_feat.items():
    if k == 'node':
        for i in v:
            N_nodes_i = (nodes['node_type'] == i).sum()
            feat_i = torch.randn(N_nodes_i, best_hyperparam['hidden_channels'])
            feat_rest[f"{i}"] = {'random': feat_i}

    elif k == 'relation':
         for i in v:
             feat_i = torch.ones(1, best_hyperparam['hidden_channels'])
             feat_rest[f"{i}"] = {'fixed': feat_i}   # I should put this also fixed instead of random, but now the result is based on random
    elif k == 'mask':
        feat_i = torch.ones(1,best_hyperparam['hidden_channels'])
        feat_rest[f"{k}"] = {'fixed': feat_i}   

    
feat.update(feat_rest)

tag = f"{model_dir}/clinical_trials/biomedformer_clinicalTrials_transductive_wl5_wpn30_dim300_head5_lencoder30_bs4000_lr0.0001_seed42_modelLastEpoch"

if len(tag) > 0:
    checkpoint_path = f"{tag}.pt"
    checkpoint_path = os.path.join('model_dir', checkpoint_path)
    checkpoint = torch.load(checkpoint_path, map_location=device)

    param = checkpoint['parameters']
    model = TransformerModel(
        vocab_size,
        best_hyperparam['hidden_channels'],
        best_hyperparam['nhead'],
        best_hyperparam['N_encoder_layers'],
        (all_param['node2vec']['walk_length']*2)-1, #walk_length_with_relation
        device=device,
        feat = feat,
        nodes = nodes,
        entity = entity,
        relation_mask_index = relation_mask_index,
        pos_emb='fixed',
    ).to(device)

    # model.load_state_dict(checkpoint['model_state_dict'], strict=False)
    
    # Original checkpoint
    state_dict = checkpoint['model_state_dict']
    model_state = model.state_dict()


    ###### debugging
    model_params = list(model.state_dict().keys())
    checkpoint_params = list(checkpoint['model_state_dict'].keys())
    
    # What parameters exist in checkpoint but not in model
    print("🔎 Params in checkpoint but not in model:", set(checkpoint_params) - set(model_params))
    
    # What parameters exist in model but not in checkpoint
    print("🔎 Params in model but not in checkpoint:", set(model_params) - set(checkpoint_params))
    
    
    # Filter out incompatible keys
    filtered_state_dict = OrderedDict()
    for k, v in state_dict.items():                
        if k in model_state and v.shape == model_state[k].shape:
            filtered_state_dict[k] = v
        # else:
        #     print(f"Skipping {k}: checkpoint shape {v.shape}, model shape {model_state.get(k, 'Not found').shape}")
    
    # Load filtered state dict
    model.load_state_dict(filtered_state_dict, strict=False)
    # optimizer = optim.Adam(model.parameters(), lr=best_hyperparam['learning_rate'])
    # optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
    start_epoch = checkpoint['epoch'] + 1
    loss = checkpoint['loss']
    
    print(start_epoch)


test_data_edge_label_index = torch.load(os.path.join(test_data_dir, "test_data_edge_label_index_with_relation_trials_random_splitted_transductive_wl5_wpn30.pt"))
test_data_edge_label = torch.load(os.path.join(test_data_dir, "test_data_edge_label_trials_random_splitted_transductive_wl5_wpn30.pt"))
edge_type_flag_test = torch.ones(len(test_data_edge_label), dtype=torch.int64)


pred_test = sim_with_logit_efficient_memory(test_data_edge_label_index, model, batch_size=best_hyperparam['batch_size'], method='prob', device=device, mask_token=mask_token, seq_len=seq_len)
pred_test = torch.tensor(pred_test, dtype=torch.float32)
pred_test


test_data = pd.DataFrame(test_data_edge_label_index.T.numpy())
test_data.columns = ['disease', 'relation', 'drug']
test_data['xy'] = test_data['disease'].astype(str) + '|' + test_data['drug'].astype(str)
test_data['label'] = test_data_edge_label
test_data['pred_test'] = pred_test

# 1. Check if the first 7327 rows of test_data['xy'] match ctg['xy']
match = test_data['xy'].iloc[:ctg.shape[0]].reset_index(drop=True).equals(ctg['xy'].reset_index(drop=True))
print("Match:", match)

test_data.loc[:ctg.shape[0]-1, 'label'] = ctg['result'].values  # use loc to avoid SettingWithCopyWarning
test_data

In [ ]:
# ── Phase and status of every trial pair ─────────────────────────────────────
PHASE = {"EARLY_PHASE1": 1, "PHASE1": 1, "PHASE1|PHASE2": 2, "PHASE2": 2, "PHASE2|PHASE3": 3, "PHASE3": 3, "PHASE4": 4}
STOPPED = {"TERMINATED", "WITHDRAWN", "SUSPENDED"}

trials = pd.read_csv(os.path.join(clinical_trials_file_results, "clinical_trial_data_with_drugbankmondo_ids.csv"),
                     usecols=["NCT Number", "Phases", "Study Status", "Study Results", "mondo_id", "drugbank_id"])
trials = trials.dropna(subset=["mondo_id", "drugbank_id"])
trials["mondo"] = trials["mondo_id"].str.split(".").str[1].astype(int).astype(str)
trials["drugbank"] = trials["drugbank_id"].str.split(".").str[1]

dis = nodes[nodes["node_type"] == "disease"]
disease_of = {m: i for nid, i in zip(dis["node_id"].astype(str), dis["node_index"]) for m in nid.split("_")}   # grouped MONDO nodes
drug_of = dict(zip(nodes.loc[nodes["node_type"] == "drug", "node_id"], nodes.loc[nodes["node_type"] == "drug", "node_index"]))
trials["disease"] = trials["mondo"].map(disease_of)
trials["drug"] = trials["drugbank"].map(drug_of)
trials = trials.dropna(subset=["disease", "drug"]).astype({"disease": int, "drug": int})
trials["phase"] = trials["Phases"].map(PHASE)

per_pair = trials.groupby(["disease", "drug"]).agg(
    max_phase=("phase", "max"), n_trials=("NCT Number", "nunique"),
    any_completed=("Study Status", lambda s: (s == "COMPLETED").any()),
    any_stopped=("Study Status", lambda s: s.isin(STOPPED).any()),
    any_results=("Study Results", lambda s: (s == "YES").any())).reset_index()

pairs = test_data.merge(per_pair, on=["disease", "drug"], how="left")
pairs["group"] = pairs["label"].map({0: "negative", 1: "clinical-trial-only", 2: "KG-supported"})
pairs.to_csv(os.path.join(OUT_DIR, "pairs_with_phase.csv"), index=False)
trial_pairs = pairs[pairs["label"] > 0]
print(f"trial pairs: {len(trial_pairs)}, with phase information: {trial_pairs['max_phase'].notna().mean():.1%}")
print(trial_pairs.groupby(["group", "max_phase"], dropna=False).size().unstack(fill_value=0))

In [ ]:
# ── Scores by highest phase and by status ────────────────────────────────────
from sklearn.metrics import roc_auc_score
from scipy.stats import spearmanr

neg = pairs.loc[pairs["label"] == 0, "pred_test"].to_numpy()


def describe(sub, name):
    s = sub["pred_test"].to_numpy()
    y = np.r_[np.ones(len(s)), np.zeros(len(neg))]
    return {"stratum": name, "pairs": len(s), "median score": np.median(s),
            "AUROC vs negatives": roc_auc_score(y, np.r_[s, neg]),
            "Mann-Whitney p (> negatives)": mannwhitneyu(s, neg, alternative="greater").pvalue}


rows = []
for grp in ["clinical-trial-only", "KG-supported"]:
    g = trial_pairs[trial_pairs["group"] == grp]
    for ph in [1, 2, 3, 4]:
        sub = g[g["max_phase"] == ph]
        if len(sub) >= 10:
            rows.append({"group": grp, **describe(sub, f"highest phase {ph}")})
    for name, m in [("any trial completed", g["any_completed"] == True), ("any trial stopped early", g["any_stopped"] == True),
                    ("results posted", g["any_results"] == True)]:
        if m.sum() >= 10:
            rows.append({"group": grp, **describe(g[m], name)})
    ok = g["max_phase"].notna()
    rho, p = spearmanr(g.loc[ok, "max_phase"], g.loc[ok, "pred_test"])
    rows.append({"group": grp, "stratum": "Spearman(highest phase, score)", "pairs": int(ok.sum()),
                 "median score": rho, "AUROC vs negatives": np.nan, "Mann-Whitney p (> negatives)": p})
res = pd.DataFrame(rows)
res.to_csv(os.path.join(OUT_DIR, "phase_stratification.csv"), index=False)
res.round(4)

## Trend test across the ordered phases

Jonckheere-Terpstra test (clinical-trial-only pairs): do the scores increase across Phase 1 < 2 < 3 < 4? It sums the
Mann-Whitney U statistics of all ordered pairs of phase groups (how often a pair from a later phase scores higher than
a pair from an earlier phase) and compares the sum with its expectation under no trend (normal approximation).
In addition: bootstrap 95% CIs of the AUROC per phase and of the difference Phase 4 - Phase 1.

In [ ]:
from scipy.stats import kendalltau, norm

t = trial_pairs[(trial_pairs["label"] == 1) & trial_pairs["max_phase"].notna()]
phases = [1, 2, 3, 4]
groups = [t.loc[t["max_phase"] == k, "pred_test"].to_numpy() for k in phases]

# Jonckheere-Terpstra: sum of U statistics over all ordered group pairs (later phase > earlier phase)
J = sum(mannwhitneyu(groups[j], groups[i]).statistic for i in range(len(groups)) for j in range(i + 1, len(groups)))
n = np.array([len(g) for g in groups]); N = n.sum()
EJ = (N ** 2 - (n ** 2).sum()) / 4
VJ = (N ** 2 * (2 * N + 3) - (n ** 2 * (2 * n + 3)).sum()) / 72
z = (J - EJ) / np.sqrt(VJ)
rho, p_rho = spearmanr(t["max_phase"], t["pred_test"])
tau, p_tau = kendalltau(t["max_phase"], t["pred_test"])
print(f"Jonckheere-Terpstra z = {z:.2f}, one-sided p = {norm.sf(z):.2e} (n = {N})")
print(f"Spearman rho = {rho:.3f}, p = {p_rho:.2e}; Kendall tau = {tau:.3f}, p = {p_tau:.2e}")

# bootstrap 95% CI of the AUROC per phase and of Phase 4 - Phase 1 (pairs and negatives resampled)
rng = np.random.default_rng(0)
auc = lambda s, ng: roc_auc_score(np.r_[np.ones(len(s)), np.zeros(len(ng))], np.r_[s, ng])
boot = {k: [] for k in phases}
for _ in range(2000):
    ng = rng.choice(neg, len(neg))
    for k, g in zip(phases, groups):
        boot[k].append(auc(rng.choice(g, len(g)), ng))
trend = pd.DataFrame([{"highest phase": k, "pairs": len(g), "AUROC": auc(g, neg),
                       "CI low": np.percentile(boot[k], 2.5), "CI high": np.percentile(boot[k], 97.5)}
                      for k, g in zip(phases, groups)])
d = np.array(boot[4]) - np.array(boot[1])
print(f"Phase 4 - Phase 1: {d.mean():.3f}, 95% CI [{np.percentile(d, 2.5):.3f}, {np.percentile(d, 97.5):.3f}]")
trend.to_csv(os.path.join(OUT_DIR, "phase_trend.csv"), index=False)
trend.round(3)

## Figure: separation from negative controls by highest trial phase and by trial status

Clinical-trial-only pairs; AUROC against the negative control pairs, bootstrap 95% CI (pairs and negatives resampled). Reads only the saved CSVs.

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import roc_auc_score

OUT_DIR = "/home/bbc8731/NetMedGPT/data/new_analysis/clinical_trials_phase"
trend = pd.read_csv(os.path.join(OUT_DIR, "phase_trend.csv"))
pp = pd.read_csv(os.path.join(OUT_DIR, "pairs_with_phase.csv"))
neg_s = pp.loc[pp["label"] == 0, "pred_test"].to_numpy()
ct = pp[pp["label"] == 1]
rng = np.random.default_rng(0)
auc = lambda s, ng: roc_auc_score(np.r_[np.ones(len(s)), np.zeros(len(ng))], np.r_[s, ng])
status = []
for name, m in [("any trial\ncompleted", ct["any_completed"] == True),
                ("any trial\nstopped early", ct["any_stopped"] == True),
                ("results\nposted", ct["any_results"] == True)]:
    s = ct.loc[m, "pred_test"].to_numpy()
    b = [auc(rng.choice(s, len(s)), rng.choice(neg_s, len(neg_s))) for _ in range(500)]
    status.append({"stratum": name, "pairs": len(s), "AUROC": auc(s, neg_s),
                   "CI low": np.percentile(b, 2.5), "CI high": np.percentile(b, 97.5)})
status = pd.DataFrame(status)

fig, axes = plt.subplots(1, 2, figsize=(10, 3.8), gridspec_kw={"width_ratios": [4, 3]})
for ax, df, labels, title in [
        (axes[0], trend, [f"Phase {int(k)}\n(n = {n:,})" for k, n in zip(trend["highest phase"], trend["pairs"])], "Highest trial phase reached"),
        (axes[1], status, [f"{s}\n(n = {n:,})" for s, n in zip(status["stratum"], status["pairs"])], "Trial status")]:
    x = np.arange(len(df))
    err = np.vstack([df["AUROC"] - df["CI low"], df["CI high"] - df["AUROC"]])
    ax.bar(x, df["AUROC"], 0.6, yerr=err, color="#D62728", capsize=3, edgecolor="black", linewidth=0.5)
    for xi, v, hi in zip(x, df["AUROC"], df["CI high"]):
        ax.text(xi, hi + 0.01, f"{v:.2f}", ha="center", va="bottom", fontsize=10)
    ax.axhline(0.5, color="grey", linewidth=0.8, linestyle="--")
    ax.set_xticks(x)
    ax.set_xticklabels(labels, fontsize=9)
    ax.set_ylim(0.4, 0.85)
    ax.set_title(title, fontsize=12)
    ax.spines[["top", "right"]].set_visible(False)
axes[0].set_ylabel("AUROC vs negative control pairs")
fig.suptitle("Clinical-trial-only drug-disease pairs: NetMedGPT separation from negative controls (95% bootstrap CI);\n"
             "trend across phases: Jonckheere-Terpstra p = 2e-23", fontsize=10)
fig.tight_layout(rect=(0, 0, 1, 0.88))
for ext in ["png", "pdf"]:
    fig.savefig(os.path.join(OUT_DIR, f"clinical_trials_phase_status.{ext}"), dpi=300, bbox_inches="tight")
plt.show()